# *Breast Cancer Detection Using Deep Learning (Fuzzy)*

# 01 - Data Exploration

Purpose:
- Load raw breast thermogram images
- Visualize sample images
- Inspect image size, channels and intensity range
- Verify dataset structure

Dataset(s):
- DMR-IR

In [ ]:
import os
import gc
import cv2
import math
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

from skimage.morphology import disk
from skimage.filters import threshold_otsu
from skimage.morphology import reconstruction

In [ ]:
print(os.getcwd())

In [ ]:
class DatasetPlot:
    # Constructor
    def __init__(self, cols, path):
        self.cols = cols
        self.path = path
        self.image_files = []

    # Images
    def imageFiles(self):
        self.image_files = [
            f for f in os.listdir(self.path)
            if f.lower().endswith(('.png', '.jpg'))
        ]
    
    # Plotting
    def imagePlotting(self):
        image_count = len(self.image_files)
        rows = math.ceil(image_count/self.cols)

        plt.figure(figsize=(self.cols*4, rows*4))
        
        for i, filename in enumerate(self.image_files):
            img_path = os.path.join(self.path, filename)
            img = mpimg.imread(img_path)
        
            plt.subplot(rows, self.cols, i+1)
            plt.imshow(img)
            plt.title(filename, fontsize=8)
            plt.axis('off')
            
        plt.tight_layout()
        plt.show()

    def main(self):
        self.imageFiles()
        self.imagePlotting()

In [ ]:
base_path = r"..\data\raw"

## 1. Thiago Alves Elias da Silva

In [ ]:
thiago_dataset = {
    "healthy": r"\Benign\226\Segmentadas",
    "diseased": r"\Malignant\255\Segmentadas",
    "mat_diseased": r"\Malignant\255\Matrizes\PAC_54_DN19.txt",
    "testing": r"\Images and Matrices from the Thesis of Thiago Alves Elias da Silva\12 New Test Cases - Testing",
    "training": r"\Images and Matrices from the Thesis of Thiago Alves Elias da Silva\Methodology Development - Training"
}
print(os.path.exists(base_path + thiago_dataset["training"] + thiago_dataset["mat_diseased"]))

### Healthy

In [ ]:
thiago_healthy_data = base_path + thiago_dataset["training"] + thiago_dataset["healthy"]

plotter1 = DatasetPlot(cols=3, path=thiago_healthy_data)
plotter1.main()

### Diseased

In [ ]:
thiago_diseased_data = base_path + thiago_dataset["training"] + thiago_dataset["diseased"]

plotter2 = DatasetPlot(cols=3, path=thiago_diseased_data)
plotter2.main()

### Matrix

In [ ]:
matrix = np.loadtxt(base_path + thiago_dataset["training"] + thiago_dataset["mat_diseased"], dtype=np.float32)
print(matrix.shape)
print(matrix.max(), matrix.min())

In [ ]:
plt.imshow(matrix, cmap="inferno")
plt.colorbar()
plt.show()

### Step A - Generate Suspicious Region (SCH-CS stage)

In [ ]:
import cv2
import numpy as np
from sklearn.cluster import KMeans

def sch_cs_segmentation(img):
    # Flatten image for clustering
    pixel_values = img.reshape((-1, 1))
    
    kmeans = KMeans(n_clusters=2)
    kmeans.fit(pixel_values)
    
    segmented = kmeans.labels_.reshape(img.shape)
    
    # Select brighter cluster
    cluster_means = [img[segmented == i].mean() for i in range(2)]
    suspicious_cluster = np.argmax(cluster_means)
    
    mask = (segmented == suspicious_cluster).astype(np.uint8)
    
    return mask


In [ ]:
kernel = np.ones((5,5), np.uint8)
mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

STEP B — Convert Mask to Initial Level Set

In [ ]:
from scipy.ndimage import distance_transform_edt

def mask_to_sdf(mask):
    inside = distance_transform_edt(mask)
    outside = distance_transform_edt(1-mask)
    sdf = inside - outside
    return sdf

In [ ]:
phi = mask_to_sdf(mask)

STEP C — Apply Level Set Refinement

In [ ]:
from skimage.segmentation import morphological_chan_vese

ls = morphological_chan_vese(img,
                             num_iter=100,
                             init_level_set=mask)

## 2. DMR-IR Different View

In [ ]:
dmr_ir_diff_view_dataset = {
    "sick": r"\DMR_IR_Different_View\Sick\0256",
    "healthy": r"\DMR_IR_Different_View\Healthy\0259"
}
print(os.path.exists(base_path + dmr_ir_diff_view_dataset["sick"]))

### Diseased

In [ ]:
plotter3 = DatasetPlot(3, base_path + dmr_ir_diff_view_dataset["sick"])
plotter3.main()

### Healthy

In [ ]:
plotter4 = DatasetPlot(3, base_path + dmr_ir_diff_view_dataset["healthy"])
plotter4.main()

## 3. Breast Cancer Dataset

In [ ]:
breast_cancer_dataset = {
    "sick": r"\breast-cancer-dataset\Train\Malignant",
    "healthy": r"\breast-cancer-dataset\Train\Benign"
}
print(os.path.exists(base_path + dmr_ir_diff_view_dataset["sick"]))

### Diseased

In [ ]:
plotter5 = DatasetPlot(3, base_path + breast_cancer_dataset["sick"])
plotter5.main()

### Healthy

In [ ]:
plotter6 = DatasetPlot(3, base_path + breast_cancer_dataset["healthy"])
plotter6.main()

## 4. Breast Thermography

In [ ]:
breast_thermography_dataset = {
    "sick": r"\Breast Thermography\Malignant\IIR0119",
    "healthy": r"\Breast Thermography\Benign\IIR0118"
}
print(os.path.exists(base_path + breast_thermography_dataset["healthy"]))

### Diseased

In [ ]:
plotter7 = DatasetPlot(3, base_path + breast_thermography_dataset["sick"])
plotter7.main()

### Healthy

In [ ]:
plotter8 = DatasetPlot(3, base_path + breast_thermography_dataset["healthy"])
plotter8.main()

## 5. BCD Dataset

In [ ]:
bcd_dataset = {
    "sick": r"\BCD_Dataset\Sick",
    "healthy": r"\BCD_Dataset\normal",
    "unknown": r"\BCD_Dataset\Unknown_class"
}
print(os.path.exists(base_path + bcd_dataset["sick"]))

### Diseased

In [ ]:
plotter9 = DatasetPlot(3, base_path + bcd_dataset["sick"])
plotter9.main()

### Healthy

In [ ]:
plotter10 = DatasetPlot(3, base_path + bcd_dataset["healthy"])
plotter10.main()

### Preprocessing - Breast Region Extraction

In [ ]:
class ImageProcessor:
    def __init__(self):
        pass

    def _load_grayscale(self, image_path):
        img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            raise ValueError(f"Could not load image from {image_path}")
        return img

    def visualize_original_processed_and_histogram(self, original_img, processed_img, original_title, processed_title, histogram_title):
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))

        # Original Image Plot
        axes[0].imshow(original_img, cmap='gray')
        axes[0].set_title(original_title)
        axes[0].axis('off')

        # Processed Image Plot
        axes[1].imshow(processed_img, cmap='gray')
        axes[1].set_title(processed_title)
        axes[1].axis('off')

        # Histogram Plot of Processed Image
        axes[2].hist(processed_img.ravel(), 256, range=[0, 256], color='gray')
        axes[2].set_title(histogram_title)
        axes[2].set_xlabel('Pixel Intensity')
        axes[2].set_ylabel('Frequency')

        plt.tight_layout()
        plt.show()

    def remove_background_soft(self, img):
        # Normalize image (important)
        img = cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX)

        # Very low threshold just to remove pure black borders
        _, thresh = cv2.threshold(img, 5, 255, cv2.THRESH_BINARY)

        # Morphological closing to fill tiny holes
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5,5))
        mask = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel)

        # Apply mask
        result = cv2.bitwise_and(img, mask)

        return result

    def remove_background_otsu(self, img):
        # otsu threshold
        _, thresh = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

        # morphological smoothing
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5,5))
        closed = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel)

        # finding largest component
        contours, _ = cv2.findContours(closed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        contours = [c for c in contours if cv2.contourArea(c) > 5000]
        if not contours:
            return np.zeros_like(img) # Return a black image if no significant contour found

        # Sort contours by area in descending order
        contours = sorted(contours, key=cv2.contourArea, reverse=True)

        mask = np.zeros_like(img)
        h, w = img.shape

        # Draw the top 2 largest contours onto the mask
        # This addresses the case where breast might be split into two main components
        for i, cnt in enumerate(contours):
            if i >= 2: # Only consider the top 2 largest contours
                break
            cv2.drawContours(mask, [cnt], -1, 255, -1)

        # Remove top 15% rows from the mask
        # Applied to the mask AFTER drawing the contours
        mask[:int(0.15*h), :] = 0

        # Applying the mask to the original image
        breast = cv2.bitwise_and(img, mask)

        return breast

    def gray_level_reconstruction(self, img):

        # Step 1: Otsu threshold
        thresh_val = threshold_otsu(img)
        binary = img > thresh_val

        # Step 2: Erode image to create marker
        kernel = disk(5)
        eroded = cv2.erode(img, np.ones((5,5), np.uint8))

        # Step 3: Reconstruction
        reconstructed = reconstruction(
            eroded,
            img,
            method='dilation'
        )

        # Convert to uint8
        reconstructed = np.uint8(reconstructed)

        return reconstructed

In [ ]:
# Initialize the image processor
processor = ImageProcessor()

# Get all image files from the directory
image_files = [
    f for f in os.listdir(base_path + bcd_dataset["sick"])
    if f.lower().endswith(('.png', '.jpg', '.jpeg'))
]
image_files.sort() # Sort to ensure consistent order

# --- USER SELECTION ---
image_selection = 'all' # @param ['all', '[0]', '[0,1]', '[0,1,2]']

# Determine which images to process based on selection
if image_selection == 'all':
    images_to_process = image_files
elif isinstance(image_selection, list):
    # Filter image_files based on provided indices
    images_to_process = [image_files[i] for i in image_selection if i < len(image_files)]
else:
    print("Invalid image_selection. Please use 'all' or a list of indices.")
    images_to_process = []

print(f"Processing {len(images_to_process)} image(s).")

# Process and visualize each selected image
for filename in images_to_process:
    img_path = os.path.join(base_path + bcd_dataset["sick"], filename)

    try:
        # Load the original image
        original_img = processor._load_grayscale(img_path)

        # Process the image (remove background)
        processed_img = processor.remove_background_soft(original_img)
        reconstructed_img = processor.gray_level_reconstruction(processed_img)

        # Visualize the comparison with histogram
        print(f"Displaying: {filename}")
        processor.visualize_original_processed_and_histogram(
            original_img, reconstructed_img,
            f"Original: {filename}",
            f"Background Removed: {filename}",
            f"Histogram of Processed: {filename}"
        )
    except ValueError as e:
        print(f"Error processing {filename}: {e}")
    except Exception as e:
        print(f"An unexpected error occurred with {filename}: {e}")

print("Image processing complete.")

# After Every Run

In [ ]:
# del df
gc.collect()